# Benchmarking
This notebook benchmarks variant effect predictors across multiple evaluation datasets. For binary-label benchmarks (clinical, functional, developmental disorder de novo, and cancer hotspots), we compute ROC AUC and threshold-based classification metrics, and estimate uncertainty via 1,000x bootstrap resampling over variants. Separate sections below implement correlation-based win-rate benchmarks for MAVE and DMS datasets.

---
### Data processing and preparation of functional, clinical, cancer, and de novo datasets
We assemble a benchmarking table containing dataset-specific labels and all available VEP scores. We also exclude VEPs with unavailable training sets.

In [ ]:
import pandas as pd
from pathlib import Path

identifiers = ["ID", "ensg"]
label_cols = ["functional_testing_label", "clinical_testing_label", "dd_label", "cancer_label"]

variant_labels = pd.read_csv("../data/intermediate/variant_labels.txt", sep="\t")
cancer = pd.read_csv("../data/datasets/cancer/cancer_hotspot_dataset.txt", sep="\t").rename(columns={"hotspot": "cancer_label"})
dd = pd.read_csv("../data/datasets/de_novo/DD_dataset.txt", sep="\t").rename(columns={"phenotype": "dd_label"})

def keep_ids_and_labels(df):
    cols = [c for c in identifiers + label_cols if c in df.columns]
    return df[cols]

variant_labels = keep_ids_and_labels(variant_labels)
cancer = keep_ids_and_labels(cancer)
dd = keep_ids_and_labels(dd)

benchmark_labels = (
    variant_labels
    .merge(dd, on=identifiers, how="outer")
    .merge(cancer, on=identifiers, how="outer")
)

func_map = {"BS3": 0, "PS3": 1}
clin_map = {"B": 0, "P": 1}
dd_map = {"DD_control": 0, "DD": 1}
benchmark_labels["functional_testing_label"] = (benchmark_labels["functional_testing_label"].map(func_map).astype("float"))
benchmark_labels["clinical_testing_label"] = (benchmark_labels["clinical_testing_label"].map(clin_map).astype("float"))
benchmark_labels["dd_label"] = (benchmark_labels["dd_label"].map(dd_map).astype("float"))
benchmark_labels["cancer_label"] = benchmark_labels["cancer_label"].astype("float")

benchmark_labels.dropna(subset=label_cols, how="all", inplace=True)

feature_matrix = pd.read_parquet(f"../data/intermediate/feature_matrix.parquet")
scores = pd.read_csv("../results/predictions/core_variant_set_scores.txt", sep="\t")
feature_matrix = feature_matrix[feature_matrix["spliceai"].isna()]

benchmark_df = benchmark_labels.merge(feature_matrix, on=identifiers, how="inner").merge(scores, on=identifiers, how="left")

primateai_path = Path("../data/restricted/primateai.parquet")
if primateai_path.exists():
    primateai = pd.read_parquet(primateai_path)
    benchmark_df = benchmark_df.merge(primateai, on=identifiers, how="left")

features_df = pd.read_csv("../resources/feature_lists/all_columns.txt", sep="\t")
veps = features_df[features_df["Type"] == "Variant Effect Predictor"]["Name"].unique().tolist()
with open("../resources/feature_lists/veps_excluded_due_to_unavailable_training_sets.txt", "r") as f:
    no_training_set = [line.strip() for line in f]
veps = [vep for vep in veps if vep not in no_training_set]

benchmark_df = benchmark_df[[col for col in (identifiers + label_cols + veps) if col in benchmark_df.columns]]

benchmark_df

,ID,ensg,functional_testing_label,clinical_testing_label,dd_label,cancer_label,FuncVEP_CTI,FuncVEP_CTE,FuncVEP_SP,ClinVEP_CTI,...,glm_AlphCaddDeogen,glm_CaddDeogenRevel,PHACT,gpnmsa_score,popEVE,ESM1v,EVH_epistatic,EVH_independent,EWSIM,sigma_score
0,1-110603569-G-A,ENSG00000177301,1.0,NaN,1.0,NaN,0.999011,0.995141,0.986611,0.999453,...,NaN,NaN,0.0,-9.68,-5.114,-13.097,-5.298,-2.736,-4.767,0.936546
1,1-110603581-G-A,ENSG00000177301,NaN,NaN,1.0,NaN,0.993176,0.972591,0.911137,0.999202,...,NaN,NaN,0.0,-11.34,-4.624,-10.555,-5.309,-0.959,-4.390,0.886448
2,1-110603893-C-T,ENSG00000177301,NaN,NaN,1.0,NaN,0.999797,0.999018,0.996939,0.999681,...,NaN,NaN,0.0,-9.18,-5.084,-11.995,-7.363,-3.514,-5.102,0.939373
3,1-110603902-C-T,ENSG00000177301,1.0,NaN,1.0,NaN,0.998806,0.997353,0.993954,0.999597,...,NaN,NaN,0.0,-10.50,-5.122,-12.286,-7.196,-3.626,-5.052,0.677924
4,1-110603914-A-G,ENSG00000177301,1.0,NaN,1.0,NaN,0.986748,0.997745,0.992842,0.998571,...,3.432472,4.068538,0.0,-11.97,-5.260,-11.914,-8.430,-4.328,-5.237,0.987695
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8612,X-77685002-C-A,ENSG00000085224,NaN,NaN,1.0,NaN,0.999270,0.999733,0.998819,0.999562,...,NaN,NaN,NaN,-10.79,NaN,NaN,NaN,NaN,NaN,NaN
8613,X-80681367-C-T,ENSG00000165288,NaN,0.0,NaN,NaN,0.020209,0.016049,0.018938,0.005175,...,-1.853469,-1.608762,NaN,-4.95,-3.862,-0.775,NaN,NaN,-2.863,0.027757
8614,X-80690090-C-A,ENSG00000165288,NaN,1.0,NaN,NaN,0.986869,0.999030,0.998699,0.999145,...,1.040154,NaN,NaN,-12.31,-4.842,-3.142,NaN,NaN,-3.791,0.963596
8615,X-85957921-T-C,ENSG00000188419,NaN,1.0,NaN,NaN,0.994014,0.998658,0.990718,0.997148,...,3.533818,NaN,NaN,-9.06,-4.723,-11.591,-7.994,-10.254,-3.935,0.637608


### Binary-label benchmarking
We evaluate each predictor using ROC AUC as the primary discrimination metric. We also report accuracy, sensitivity, and specificity at the threshold that maximizes Youden’s J statistic (sensitivity + specificity − 1). To quantify uncertainty, we bootstrap AUC by resampling variants with replacement 1,000 times.

We cannot apply gene-level win-rate benchmarking here since it requires enough variants per gene to compute per-gene AUCs reliably. In the temporally held-out ClinVar testing set, only 15 genes have ≥10 benign and ≥10 pathogenic variants, even without balancing. The DD de novo set has no genes meeting this minimum, and the cancer set has only 5. Because this yields unstable win-rate estimates, we use variant-level AUC with bootstrap uncertainty for these binary-label datasets, and apply win-rate aggregation only for MAVE and DMS datasets. Likewise, applying gene family-based win-rate benchmarking on the functional testing set causes very high variance due to a low number of families.


In [6]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

def compute_binary_metrics(y, scores, min_pos=None, min_neg=None, fixed_direction=None):
    y = np.asarray(y).astype(int)
    scores = np.asarray(scores).astype(float)

    # Need both classes
    if np.unique(y).size < 2:
        return None

    # Need score variation
    if np.unique(scores).size < 2:
        return None

    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())

    if min_pos is not None and n_pos < min_pos:
        return None
    if min_neg is not None and n_neg < min_neg:
        return None

    auc_orig = roc_auc_score(y, scores)
    auc_adj = max(auc_orig, 1 - auc_orig)

    if fixed_direction in ("higher", "lower"):
        direction = fixed_direction
    else:
        direction = "higher" if auc_orig >= 0.5 else "lower"

    scores_for_roc = scores if direction == "higher" else -scores

    fpr, tpr, thresholds = roc_curve(y, scores_for_roc)

    # Drop infinite thresholds
    finite_mask = np.isfinite(thresholds)
    fpr = fpr[finite_mask]
    tpr = tpr[finite_mask]
    thresholds = thresholds[finite_mask]

    if thresholds.size == 0:
        return None

    j = tpr - fpr
    best_idx = int(np.argmax(j))

    thresh_transformed = float(thresholds[best_idx])

    if direction == "higher":
        thresh_orig = thresh_transformed
        y_pred = (scores >= thresh_orig).astype(int)
    else:
        thresh_orig = -thresh_transformed
        y_pred = (scores <= thresh_orig).astype(int)

    sens = float(tpr[best_idx])
    spec = float(1 - fpr[best_idx])
    youden = float(j[best_idx])
    acc = float(accuracy_score(y, y_pred))

    return {
        "AUC": float(auc_adj),
        "Accuracy": acc,
        "OptimizedThreshold": float(thresh_orig),
        "Sensitivity": sens,
        "Specificity": spec,
        "YoudensJ": youden,
        "PathogenicDirection": direction,
        "n_pos": n_pos,
        "n_neg": n_neg,
    }

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

def benchmark_veps_for_label(df, label_col, vep_cols, min_variants_per_class=50, n_bootstrap=1000, bootstrap_random_state=42):
    df = df.copy()
    df = df.dropna(subset=[label_col])

    results = []
    vep_cols = [v for v in vep_cols if v in df.columns]

    rng = np.random.RandomState(bootstrap_random_state)

    for vep in vep_cols:
        sub = df[[label_col, vep]].dropna()
        if sub.empty:
            results.append({"VEP": vep, "AUC": np.nan, "AUC_boot_mean": np.nan, "AUC_boot_std": np.nan, "Accuracy": np.nan, "OptimizedThreshold": np.nan, 
                            "Sensitivity": np.nan, "Specificity": np.nan, "YoudensJ": np.nan, "PathogenicDirection": np.nan, "n_pos": 0, "n_neg": 0})
            continue

        y = sub[label_col].astype(int).values
        scores = sub[vep].astype(float).values
        n_pos = int((y == 1).sum())
        n_neg = int((y == 0).sum())

        # Pre-filter based on class size
        if n_pos < min_variants_per_class or n_neg < min_variants_per_class:
            results.append({"VEP": vep, "AUC": np.nan, "AUC_boot_mean": np.nan, "AUC_boot_std": np.nan, "Accuracy": np.nan, "OptimizedThreshold": np.nan, 
                                "Sensitivity": np.nan, "Specificity": np.nan, "YoudensJ": np.nan, "PathogenicDirection": np.nan, "n_pos": n_pos, "n_neg": n_neg})
            continue

        metrics = compute_binary_metrics(y, scores, min_pos=min_variants_per_class, min_neg=min_variants_per_class, fixed_direction=None)

        if metrics is None:
            results.append({"VEP": vep, "AUC": np.nan, "AUC_boot_mean": np.nan, "AUC_boot_std": np.nan, "Accuracy": np.nan, "OptimizedThreshold": np.nan, 
                                    "Sensitivity": np.nan, "Specificity": np.nan, "YoudensJ": np.nan, "PathogenicDirection": np.nan, "n_pos": n_pos, "n_neg": n_neg})
            continue

        auc_point = metrics["AUC"]
        acc = metrics["Accuracy"]
        thresh = metrics["OptimizedThreshold"]
        sens = metrics["Sensitivity"]
        spec = metrics["Specificity"]
        youden = metrics["YoudensJ"]
        direction = metrics["PathogenicDirection"]

        # Bootstrap AUC
        boot_aucs = []
        n = len(y)

        for _ in range(n_bootstrap):
            idx = rng.randint(0, n, size=n)
            y_b = y[idx]
            s_b = scores[idx]

            if np.unique(y_b).size < 2:
                continue

            try:
                auc_raw = roc_auc_score(y_b, s_b)
            except ValueError:
                continue

            auc_adj_b = max(auc_raw, 1 - auc_raw)
            boot_aucs.append(auc_adj_b)

        if boot_aucs:
            auc_boot_mean = float(np.mean(boot_aucs))
            auc_boot_std = float(np.std(boot_aucs, ddof=1))
        else:
            auc_boot_mean = np.nan
            auc_boot_std = np.nan

        results.append(
            {
                "VEP": vep,
                "AUC": auc_point,
                "AUC_boot_mean": auc_boot_mean,
                "AUC_boot_std": auc_boot_std,
                "Accuracy": acc,
                "OptimizedThreshold": thresh,
                "Sensitivity": sens,
                "Specificity": spec,
                "YoudensJ": youden,
                "PathogenicDirection": direction,
                "n_pos": n_pos,
                "n_neg": n_neg,
            }
        )

    if not results:
        return pd.DataFrame(columns=["VEP", "AUC", "AUC_boot_mean", "AUC_boot_std", "Accuracy", "OptimizedThreshold", "Sensitivity", "Specificity", "YoudensJ", "PathogenicDirection", "n_pos", "n_neg"])

    df_out = pd.DataFrame(results)
    df_out = df_out.sort_values("AUC", ascending=False).reset_index(drop=True)
    return df_out

We run the binary-label benchmarking procedure for the clinical testing set, functional testing set, developmental disorder de novo dataset, and cancer hotspot dataset. Predictors are included only if they have sufficient scored variants in both classes after filtering.


In [4]:
vep_cols = [c for c in benchmark_df.columns if c not in identifiers + label_cols]

os.makedirs("../results/benchmarks/", exist_ok=True)

clinical_bench = benchmark_veps_for_label(benchmark_df, label_col="clinical_testing_label", vep_cols=vep_cols, min_variants_per_class=100, n_bootstrap=1000, bootstrap_random_state=42).to_csv("../results/benchmarks/clinical_benchmark.txt", sep="\t", index=False)

dd_bench = benchmark_veps_for_label(benchmark_df, label_col="dd_label", vep_cols=vep_cols, min_variants_per_class=50, n_bootstrap=1000, bootstrap_random_state=42).to_csv("../results/benchmarks/dd_benchmark.txt", sep="\t", index=False)

cancer_bench = benchmark_veps_for_label(benchmark_df, label_col="cancer_label", vep_cols=vep_cols, min_variants_per_class=100, n_bootstrap=1000, bootstrap_random_state=42).to_csv("../results/benchmarks/cancer_benchmark.txt", sep="\t", index=False)

functional_bench = benchmark_veps_for_label(benchmark_df, label_col="functional_testing_label", vep_cols=vep_cols, min_variants_per_class=100, n_bootstrap=1000, bootstrap_random_state=42).to_csv("../results/benchmarks/functional_benchmark.txt", sep="\t", index=False)

### Processing and preparation of MAVE and ProteinGym DMS datasets
We assemble two quantitative functional score datasets for correlation-based benchmarking. 

1) **Minimum study size**: we retain only MAVE/DMS studies with at least 1,000 unique missense variants.

2) **One study per gene**: for genes with multiple eligible studies, we keep the study with the largest number of missense variants for that gene.

An alternative approach is to select, per protein, the study with the highest mean correlation across all benchmarked VEPs. We avoid this because it can bias study selection toward datasets that agree best with the particular mix of predictors evaluated (e.g., many clinical-trained models), rather than providing a predictor-agnostic summary of assay evidence.

In [ ]:
import pandas as pd
from pathlib import Path

identifiers = ["ID", "ensg"]

mave = pd.read_csv("../data/datasets/functional/mave_without_calibration.txt", sep="\t", low_memory=False)
mave["ID"] = mave[["chr", "pos", "ref", "alt"]].astype(str).agg("-".join, axis=1)
mave = mave[identifiers + ["score", "source"]]
mave.dropna(subset="score", inplace=True)

feature_matrix = pd.read_parquet(f"../data/intermediate/feature_matrix.parquet")
scores = pd.read_csv("../results/predictions/core_variant_set_scores.txt", sep="\t")
non_missense = pd.read_parquet("../data/intermediate/non_missense_variants.parquet")
feature_matrix = (feature_matrix.merge(non_missense[identifiers].drop_duplicates(), on=identifiers, how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge"))
feature_matrix = feature_matrix[feature_matrix["spliceai"].isna()]

mave = mave.merge(feature_matrix, on=identifiers, how="inner").merge(scores, on=identifiers, how="left")

primateai_path = Path("../data/restricted/primateai.parquet")
if primateai_path.exists():
    primateai = pd.read_parquet(primateai_path)
    mave = mave.merge(primateai, on=identifiers, how="left")

features_df = pd.read_csv("../resources/feature_lists/all_columns.txt", sep="\t")
veps = features_df[features_df["Type"] == "Variant Effect Predictor"]["Name"].unique().tolist()
with open("../resources/feature_lists/veps_excluded_due_to_unavailable_training_sets.txt", "r") as f:
    no_training_set = [line.strip() for line in f]
veps = [vep for vep in veps if vep not in no_training_set]

mave = mave[[col for col in (identifiers + ["score", "source"] + veps) if col in mave.columns]]

proteingym = pd.read_csv("../data/datasets/proteingym/proteingym_dms.txt", sep="\t")
proteingym = proteingym[(proteingym["canonical_seq_match"] == 1) & (proteingym["snv_missense_mapped"] == 1)].dropna(subset=["ID"])
proteingym = proteingym.merge(pd.read_parquet("../data/intermediate/proteingym_feature_matrix.parquet"), on=identifiers, how="inner")
proteingym = proteingym.merge(pd.read_csv("../results/predictions/proteingym_scores.txt", sep="\t"), on=identifiers, how="left")
proteingym = (proteingym.merge(non_missense[identifiers].drop_duplicates(), on=identifiers, how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge"))
proteingym = proteingym[proteingym["spliceai"].isna()]

pg_primateai_path = Path("../data/restricted/proteingym_primateai.parquet")
if pg_primateai_path.exists():
    pg_primateai = pd.read_parquet(pg_primateai_path)
    proteingym = proteingym.merge(pg_primateai, on=identifiers, how="left")

proteingym = proteingym[[col for col in (identifiers + ["DMS_score", "study_id"] + veps) if col in proteingym.columns]]

def filter_studies_by_size(df, study_col, min_variants=1000, id_col="ID"):
    counts = df.groupby(study_col)[id_col].nunique()
    keep_studies = counts[counts >= min_variants].index
    df_filtered = df[df[study_col].isin(keep_studies)].copy()
    return df_filtered

mave = filter_studies_by_size(mave, study_col="source", min_variants=1000)
proteingym = filter_studies_by_size(proteingym, study_col="study_id", min_variants=1000)

def keep_largest_study_per_gene(df, gene_col="ensg", study_col="source", id_col="ID"):
    counts = (df.groupby([gene_col, study_col])[id_col].nunique().reset_index(name="n_variants"))
    best_per_gene = (counts.sort_values(["n_variants"], ascending=False).drop_duplicates(subset=[gene_col])[[gene_col, study_col]])
    df_filtered = df.merge(best_per_gene, on=[gene_col, study_col], how="inner")
    return df_filtered

mave = keep_largest_study_per_gene(mave, gene_col="ensg", study_col="source",   id_col="ID")
proteingym = keep_largest_study_per_gene(proteingym, gene_col="ensg", study_col="study_id", id_col="ID")

### Correlation-based win-rate benchmarking (rank score)
For MAVE/DMS benchmarks, predictors are evaluated within each gene/protein by agreement with experimental measurements. For each gene, we compare predictors pairwise using the absolute Spearman correlation between predictor scores and the DMS score.

To ensure fair comparisons under missingness, each pairwise correlation is computed on the intersection of variants for which the DMS score and both predictors have non-missing values. A pairwise comparison for a gene is performed only when this shared set contains at least 500 variants.

Within each gene, the predictor with the higher absolute correlation is counted as a win for that gene. Wins are aggregated across genes to obtain pairwise win rates, and each predictor is summarized by a rank score, defined as its average win rate against all other predictors.

In [6]:
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def benchmark_corr_winrate_by_unit(
    dataset,
    tool_columns,
    score_column,
    unit_column,
    min_variants_per_gene=500,
    min_genes_per_tool=10,
    n_bootstraps=1000,
    random_state=42,
    min_units_per_pair=None,
    min_opponents_per_tool=None,
    out_summary_path=None,
    out_pairwise_path=None,
    out_rankwins_path=None,
):
    df = dataset.copy()
    df = df.dropna(subset=[score_column, unit_column])
    df[score_column] = df[score_column].astype(float)

    tool_columns = [t for t in tool_columns if t in df.columns]
    if not tool_columns:
        empty_summary = pd.DataFrame(
            columns=["VEP", "RankScore", "RankScoreSD", "Proteins", "MeanCorrelation", "NumOpponentsUsed"])
        empty_pairwise = pd.DataFrame()
        return empty_summary, empty_pairwise

    genes = sorted(df[unit_column].dropna().unique())
    veps = sorted(tool_columns)
    n_tools = len(veps)
    n_genes = len(genes)
    tool_to_idx = {v: i for i, v in enumerate(veps)}

    full_win_counts = np.zeros((n_tools, n_tools), dtype=float)
    full_pair_counts = np.zeros((n_tools, n_tools), dtype=int)

    gene_win_counts_list = []
    gene_pair_counts_list = []

    tool_gene_corr_values = {v: [] for v in veps}
    genes_per_tool = {v: 0 for v in veps}

    for gene in genes:
        gdf = df[df[unit_column] == gene]
        cols = [score_column] + veps
        gdf = gdf[cols].copy()
        gdf = gdf.dropna(subset=[score_column])
        if gdf.empty:
            gene_win_counts_list.append(np.zeros((n_tools, n_tools), dtype=float))
            gene_pair_counts_list.append(np.zeros((n_tools, n_tools), dtype=int))
            continue

        y_all = gdf[score_column].astype(float).values

        gene_win = np.zeros((n_tools, n_tools), dtype=float)
        gene_pairs = np.zeros((n_tools, n_tools), dtype=int)

        corr_per_tool_this_gene = {v: [] for v in veps}

        for i in range(n_tools):
            ti = veps[i]
            xi_all = gdf[ti].astype(float).values

            for j in range(i + 1, n_tools):
                tj = veps[j]
                xj_all = gdf[tj].astype(float).values

                mask = np.isfinite(y_all) & np.isfinite(xi_all) & np.isfinite(xj_all)
                n = int(mask.sum())
                if n < min_variants_per_gene:
                    continue

                y = y_all[mask]
                xi = xi_all[mask]
                xj = xj_all[mask]

                if np.nanvar(xi) == 0 or np.nanvar(xj) == 0 or np.nanvar(y) == 0:
                    continue

                rho_i, _ = spearmanr(xi, y)
                rho_j, _ = spearmanr(xj, y)
                corr_i = float(abs(rho_i)) if np.isfinite(rho_i) else np.nan
                corr_j = float(abs(rho_j)) if np.isfinite(rho_j) else np.nan

                if not (np.isfinite(corr_i) and np.isfinite(corr_j)):
                    continue

                corr_per_tool_this_gene[ti].append(corr_i)
                corr_per_tool_this_gene[tj].append(corr_j)

                if corr_i > corr_j:
                    gene_win[i, j] += 1.0
                    gene_pairs[i, j] += 1
                elif corr_j > corr_i:
                    gene_win[j, i] += 1.0
                    gene_pairs[i, j] += 1

        full_win_counts += gene_win
        full_pair_counts += gene_pairs
        gene_win_counts_list.append(gene_win)
        gene_pair_counts_list.append(gene_pairs)

        for t in veps:
            vals = corr_per_tool_this_gene[t]
            if len(vals) > 0:
                genes_per_tool[t] += 1
                tool_gene_corr_values[t].append(float(np.nanmean(vals)))

    if n_genes == 0 or np.all(full_pair_counts == 0):
        empty_summary = pd.DataFrame(
            columns=["VEP", "RankScore", "RankScoreSD", "Proteins", "MeanCorrelation", "NumOpponentsUsed"])
        empty_pairwise = pd.DataFrame()
        return empty_summary, empty_pairwise

    mean_corr_per_tool = np.array(
        [
            np.nanmean(tool_gene_corr_values[v]) if len(tool_gene_corr_values[v]) > 0 else np.nan
            for v in veps
        ],
        dtype=float,
    )
    genes_per_tool_arr = np.array([genes_per_tool[v] for v in veps], dtype=int)

    if min_units_per_pair is not None:
        pair_mask = full_pair_counts >= min_units_per_pair
    else:
        pair_mask = np.ones_like(full_pair_counts, dtype=bool)

    pair_mask = np.logical_or(pair_mask, pair_mask.T)
    np.fill_diagonal(pair_mask, False)

    def compute_rank_scores_from_counts(win_counts, pair_counts, pair_mask):
        win_rate = np.full_like(win_counts, np.nan, dtype=float)

        valid_pairs = (pair_counts > 0) & pair_mask
        with np.errstate(divide="ignore", invalid="ignore"):
            win_rate[valid_pairs] = win_counts[valid_pairs] / pair_counts[valid_pairs]

        for i in range(n_tools):
            for j in range(i + 1, n_tools):
                if valid_pairs[i, j]:
                    win_rate[j, i] = 1.0 - win_rate[i, j]

        rank_scores = []
        for i in range(n_tools):
            row = win_rate[i, :]
            valid = (~np.isnan(row)) & pair_mask[i, :]
            valid[i] = False
            if not np.any(valid):
                rs = np.nan
            else:
                rs = float(np.nanmean(row[valid]))
            rank_scores.append(rs)

        return np.array(rank_scores, dtype=float), win_rate

    full_rank_scores, full_win_rate = compute_rank_scores_from_counts(
        full_win_counts,
        full_pair_counts,
        pair_mask,
    )

    if n_genes < 2 or n_bootstraps <= 0:
        boot_rank_scores = np.full((0, n_tools), np.nan)
        rank_win_counts = np.zeros((n_tools, n_tools), dtype=int)
        rank_comp_counts = np.zeros((n_tools, n_tools), dtype=int)
    else:
        rng = np.random.RandomState(random_state)
        boot_rank_scores = np.zeros((n_bootstraps, n_tools), dtype=float)

        gene_win_counts_arr = np.stack(gene_win_counts_list, axis=0)
        gene_pair_counts_arr = np.stack(gene_pair_counts_list, axis=0)

        rank_win_counts = np.zeros((n_tools, n_tools), dtype=int)
        rank_comp_counts = np.zeros((n_tools, n_tools), dtype=int)

        for b in range(n_bootstraps):
            boot_idx = rng.randint(0, n_genes, size=n_genes)
            boot_win = gene_win_counts_arr[boot_idx].sum(axis=0)
            boot_pair = gene_pair_counts_arr[boot_idx].sum(axis=0)

            rs_boot, _ = compute_rank_scores_from_counts(
                boot_win,
                boot_pair,
                pair_mask,
            )
            boot_rank_scores[b, :] = rs_boot

            for i in range(n_tools):
                rsi = rs_boot[i]
                if not np.isfinite(rsi):
                    continue
                for j in range(n_tools):
                    if i == j:
                        continue
                    rsj = rs_boot[j]
                    if not np.isfinite(rsj):
                        continue
                    rank_comp_counts[i, j] += 1
                    if rsi > rsj:
                        rank_win_counts[i, j] += 1

    if boot_rank_scores.shape[0] > 0:
        rank_score_sd = np.nanstd(boot_rank_scores, axis=0, ddof=0)
    else:
        rank_score_sd = np.full(n_tools, np.nan, dtype=float)

    opponents_used = np.sum(pair_mask, axis=1)

    summary_df = pd.DataFrame(
        {
            "VEP": veps,
            "RankScore": full_rank_scores,
            "RankScoreSD": rank_score_sd,
            "Proteins": genes_per_tool_arr,
            "MeanCorrelation": mean_corr_per_tool,
            "NumOpponentsUsed": opponents_used,
        }
    )

    mask_few_genes = summary_df["Proteins"] < min_genes_per_tool

    if min_opponents_per_tool is not None:
        mask_few_opponents = summary_df["NumOpponentsUsed"] < min_opponents_per_tool
    else:
        mask_few_opponents = np.zeros(len(summary_df), dtype=bool)

    mask_bad = mask_few_genes | mask_few_opponents
    summary_df.loc[mask_bad, ["RankScore", "RankScoreSD", "MeanCorrelation"]] = np.nan

    summary_df = summary_df.sort_values(
        "RankScore", ascending=False, na_position="last"
    ).reset_index(drop=True)

    rank_order = summary_df.loc[summary_df["RankScore"].notna(), "VEP"].tolist()
    pairwise_df = pd.DataFrame(full_win_rate, index=veps, columns=veps)
    if rank_order:
        pairwise_df = pairwise_df.loc[rank_order, rank_order]
    else:
        pairwise_df = pairwise_df.iloc[0:0, 0:0]

    if out_summary_path is not None:
        os.makedirs(os.path.dirname(out_summary_path), exist_ok=True)
        summary_df.to_csv(out_summary_path, sep="\t", index=False)
        print(f"Saved win-rate summary to {out_summary_path}")

    if out_pairwise_path is not None:
        os.makedirs(os.path.dirname(out_pairwise_path), exist_ok=True)
        pairwise_df.to_csv(out_pairwise_path, sep="\t")
        print(f"Saved pairwise win-rate matrix to {out_pairwise_path}")

    if out_rankwins_path is not None and n_bootstraps > 0:
        rows = []
        rank_order = summary_df.loc[summary_df["RankScore"].notna(), "VEP"].tolist()
        for vi in rank_order:
            i = tool_to_idx[vi]
            for vj in rank_order:
                if vi == vj:
                    continue
                j = tool_to_idx[vj]
                n_comp = rank_comp_counts[i, j]
                if n_comp == 0:
                    continue
                n_wins = rank_win_counts[i, j]
                win_rate = n_wins / n_comp
                rows.append(
                    {
                        "VEP_higher_rank": vi,
                        "VEP_lower_rank": vj,
                        "n_bootstraps_higher_rankscore": int(n_wins),
                        "n_bootstraps_compared": int(n_comp),
                        "higher_ranking_proportion": float(win_rate),
                    }
                )

        rankwins_df = pd.DataFrame(rows)
        os.makedirs(os.path.dirname(out_rankwins_path), exist_ok=True)
        rankwins_df.to_csv(out_rankwins_path, sep="\t", index=False)
        print(f"Saved rank-score win-count table to {out_rankwins_path}")

    return summary_df, pairwise_df


We apply additional robustness criteria so that rank scores reflect stable comparisons across genes and across methods:

- **Gene coverage per predictor**: a predictor is included in the final ranking only if it has usable correlation comparisons in at least half of genes.

- **Support per predictor pair**: a pairwise win rate between two predictors is computed only if their comparison is supported by at least half of genes.

- **Opponent coverage**: a predictor must have valid pairwise comparisons against at least half of other predictors.

Rank-score uncertainty is estimated by 1,000x bootstrap resampling over genes.


In [9]:
vep_cols_mave = [c for c in mave.columns if c not in identifiers + ["score", "source"]]
vep_cols_pg = [c for c in proteingym.columns if c not in identifiers + ["DMS_score", "study_id"]]

n_genes_mave = mave["ensg"].nunique()
min_genes_per_tool_mave = int(np.ceil(n_genes_mave / 2))
min_units_per_pair_mave = int(np.ceil(n_genes_mave / 2))
n_tools_mave = len(vep_cols_mave)
min_opponents_per_tool_mave = int(np.ceil(n_tools_mave / 2))

n_genes_pg = proteingym["ensg"].nunique()
min_genes_per_tool_pg = int(np.ceil(n_genes_pg / 2))
min_units_per_pair_pg = int(np.ceil(n_genes_pg / 2))
n_tools_pg = len(vep_cols_pg)
min_opponents_per_tool_pg = int(np.ceil(n_tools_pg / 2))

mave_win, mave_pair = benchmark_corr_winrate_by_unit(
    dataset=mave,
    tool_columns=vep_cols_mave,
    score_column="score",
    unit_column="ensg",
    min_variants_per_gene=500,
    min_genes_per_tool=min_genes_per_tool_mave,
    n_bootstraps=1000,
    random_state=42,
    min_units_per_pair=min_units_per_pair_mave,
    min_opponents_per_tool=min_opponents_per_tool_mave,
    out_summary_path="../results/benchmarks/mave_benchmark.txt",
    out_pairwise_path="../results/benchmarks/mave_benchmark_pairwise_winrate.txt",
    out_rankwins_path="../results/benchmarks/mave_ranking_wins.txt",
)

pg_win, pg_pair = benchmark_corr_winrate_by_unit(
    dataset=proteingym,
    tool_columns=vep_cols_pg,
    score_column="DMS_score",
    unit_column="ensg",
    min_variants_per_gene=500,
    min_genes_per_tool=min_genes_per_tool_pg,
    n_bootstraps=1000,
    random_state=42,
    min_units_per_pair=min_units_per_pair_pg,
    min_opponents_per_tool=min_opponents_per_tool_pg,
    out_summary_path="../results/benchmarks/proteingym_benchmark.txt",
    out_pairwise_path="../results/benchmarks/proteingym_benchmark_pairwise_winrate.txt",
    out_rankwins_path="../results/benchmarks/proteingym_ranking_wins.txt",
)

/home/tozcelik3/miniconda3/envs/funcvep/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Saved win-rate summary to ../results/benchmarks/mave_benchmark.txt
Saved pairwise win-rate matrix to ../results/benchmarks/mave_benchmark_pairwise_winrate.txt
Saved rank-score win-count table to ../results/benchmarks/mave_ranking_wins.txt
Saved win-rate summary to ../results/benchmarks/proteingym_benchmark.txt
Saved pairwise win-rate matrix to ../results/benchmarks/proteingym_benchmark_pairwise_winrate.txt
Saved rank-score win-count table to ../results/benchmarks/proteingym_ranking_wins.txt


/home/tozcelik3/miniconda3/envs/funcvep/lib/python3.9/site-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [4]:
import pandas as pd

mave_ids = mave["ID"].astype(str)
mave_ensg = mave["ensg"].astype(str)
pg_ids = proteingym["ID"].astype(str)
pg_ensg = proteingym["ensg"].astype(str)
mave_genes = set(mave_ensg.dropna().unique())
pg_genes = set(pg_ensg.dropna().unique())

genes_intersection = mave_genes & pg_genes
genes_only_mave = mave_genes - pg_genes
genes_only_pg = pg_genes - mave_genes

print("Gene-level overlap")
print(f"MAVE genes: {len(mave_genes)}")
print(f"ProteinGym genes: {len(pg_genes)}")
print(f"Overlap genes: {len(genes_intersection)}")
print(f"Only in MAVE: {len(genes_only_mave)}")
print(f"Only in ProteinGym: {len(genes_only_pg)}")
print(f"Frac of MAVE genes that appear in PG: {len(genes_intersection) / len(mave_genes):.4f}")
print(f"Frac of PG genes that appear in MAVE: {len(genes_intersection) / len(pg_genes):.4f}")

mave_pairs = set(zip(mave_ids, mave_ensg))
pg_pairs = set(zip(pg_ids, pg_ensg))

pairs_intersection = mave_pairs & pg_pairs
pairs_only_mave = mave_pairs - pg_pairs
pairs_only_pg = pg_pairs - mave_pairs

print("\nVariant-level overlap")
print(f"MAVE (ID, ensg) pairs: {len(mave_pairs)}")
print(f"ProteinGym (ID, ensg) pairs: {len(pg_pairs)}")
print(f"Overlap pairs: {len(pairs_intersection)}")
print(f"Only in MAVE: {len(pairs_only_mave)}")
print(f"Only in ProteinGym: {len(pairs_only_pg)}")
print(f"Frac of MAVE pairs in PG: {len(pairs_intersection) / len(mave_pairs):.6f}")
print(f"Frac of PG pairs in MAVE: {len(pairs_intersection) / len(pg_pairs):.6f}")


Gene-level overlap
MAVE genes: 31
ProteinGym genes: 27
Overlap genes: 16
Only in MAVE: 15
Only in ProteinGym: 11
Frac of MAVE genes that appear in PG: 0.5161
Frac of PG genes that appear in MAVE: 0.5926

Variant-level overlap
MAVE (ID, ensg) pairs: 88286
ProteinGym (ID, ensg) pairs: 65714
Overlap pairs: 40243
Only in MAVE: 48043
Only in ProteinGym: 25471
Frac of MAVE pairs in PG: 0.455825
Frac of PG pairs in MAVE: 0.612396
